In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from envs.cuas_env import CUASEnv
env = CUASEnv()
obs, info = env.reset(seed=42)
print('Environment created successfully!')
print('Observation keys:', list(obs.keys()))
print('n_interceptors:', env.n_interceptors)
print('n_enemy_drones:', env.n_enemy_drones)

In [ ]:
# Run 2 full episodes
episode_rewards = []
for ep in range(2):
    obs, _ = env.reset(seed=ep)
    ep_reward = 0
    for t in range(50):
        actions = {
            'commander': np.zeros(env.n_interceptors, dtype=int),
            **{f'interceptor_{i}': np.random.uniform(-1, 1, 3).astype('float32')
               for i in range(env.n_interceptors)}
        }
        obs, rewards, terminated, truncated, info = env.step(actions)
        ep_reward += sum(rewards.values())
        if terminated or truncated:
            break
    episode_rewards.append(ep_reward)
    print(f'Episode {ep+1} reward: {ep_reward:.2f}, steps: {env.current_step}')
print('Two episodes completed')

In [ ]:
# Render episode frame
env.reset(seed=42)
for _ in range(10):
    actions = {
        'commander': np.zeros(env.n_interceptors, dtype=int),
        **{f'interceptor_{i}': np.random.uniform(-1, 1, 3).astype('float32')
           for i in range(env.n_interceptors)}
    }
    env.step(actions)
fig = env.render()
plt.savefig('/tmp/env_frame.png', dpi=80, bbox_inches='tight')
print('Frame rendered and saved')
plt.close('all')

In [ ]:
# Plot reward curve per timestep
rewards_over_time = []
env.reset(seed=0)
for t in range(100):
    actions = {
        'commander': np.zeros(env.n_interceptors, dtype=int),
        **{f'interceptor_{i}': np.random.uniform(-0.5, 0.5, 3).astype('float32')
           for i in range(env.n_interceptors)}
    }
    _, rewards, terminated, truncated, _ = env.step(actions)
    rewards_over_time.append(sum(rewards.values()))
    if terminated or truncated:
        break
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(rewards_over_time)
ax.set_xlabel('Timestep')
ax.set_ylabel('Total Reward')
ax.set_title('Reward per Timestep')
ax.grid(True)
plt.tight_layout()
plt.savefig('/tmp/reward_curve.png', dpi=80)
plt.close('all')
print('Reward curve saved')

In [ ]:
# Show jammer zones + GAT edges on grid
import matplotlib.patches as patches
env.reset(seed=42)
fig, ax = plt.subplots(figsize=(8, 8))
ax.set_xlim(0, env.grid_size)
ax.set_ylim(0, env.grid_size)
ax.set_facecolor('#1a1a2e')
# Jammer zones
for jammer in env.hw_jammers:
    circle = patches.Circle(jammer.position[:2], jammer.radius,
                             color='red', alpha=0.3, fill=True)
    ax.add_patch(circle)
# Interceptors and GAT edges
positions = env.interceptor_positions
from models.gat_network import SwarmGAT
gat = SwarmGAT()
edge_index, edge_attr = gat.build_adjacency(positions, comm_radius=env.comm_radius)
for k in range(edge_index.shape[1]):
    i, j = edge_index[0, k].item(), edge_index[1, k].item()
    ax.plot([positions[i, 0], positions[j, 0]], [positions[i, 1], positions[j, 1]],
            'purple', alpha=0.5, lw=1)
for i, pos in enumerate(positions):
    ax.scatter(pos[0], pos[1], c='cyan', s=120, marker='^', zorder=5)
ax.set_title('Jammer Zones + GAT Communication Edges', color='white')
plt.savefig('/tmp/jammer_gat.png', dpi=80)
plt.close('all')
print('Jammer + GAT visualization saved')

In [ ]:
# Episode summary metrics
from evaluation.metrics import swarm_neutralization_rate, friendly_fire_rate
print('=== Episode Summary ===')
print(f'Neutralized: {env.neutralized_count}/{env.n_enemy_drones}')
print(f'Neutralization Rate: {swarm_neutralization_rate(env.neutralized_count, env.n_enemy_drones):.1f}%')
print(f'Friendly Fire: {env.friendly_fire_count}')
print(f'Steps: {env.current_step}/{env.max_timesteps}')
print(f'Alive drones: {env.swarm.n_alive if env.swarm else 0}')